# XGBoost Training — Safe Lag v3

Retrain XGBoost với **safe lag v3 feature set** (từ `splits_safe_lag_v3/`):

| | v2 (cũ) | v3 (mới) |
|---|---|---|
| Removed | 6 toxic lags | 8 (thêm rolling_mean_28, rolling_std_7) |
| Added | lag_16, rolling_mean_30 | lag_16/21/35, rolling_mean_30/28_safe, rolling_std_28_safe |
| Features | 45 | 47 |
| Rolling shift | shift(1) → distribution shift | shift(16) → no distribution shift |

**GPU**: `tree_method='hist'` + `device='cuda'`  
**Params**: Optuna best params từ lần tuning trước

## 1. Setup & Load Data

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'
SPLITS_V3     = PROCESSED_DIR / 'splits_safe_lag_v3'

assert SPLITS_V3.exists(), (
    f'splits_safe_lag_v3 không tồn tại: {SPLITS_V3}\n'
    f'Hãy chạy feature_safe_lag_v3.ipynb trước!'
)
print(f'SPLITS_V3  : {SPLITS_V3}')
print(f'MODELS_DIR : {MODELS_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import joblib
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_log_error, mean_squared_error, mean_absolute_error

print('Loading splits từ splits_safe_lag_v3/...')
X_train    = pd.read_csv(SPLITS_V3 / 'train_features.csv')
y_train    = pd.read_csv(SPLITS_V3 / 'train_target.csv')
X_val      = pd.read_csv(SPLITS_V3 / 'val_features.csv')
y_val      = pd.read_csv(SPLITS_V3 / 'val_target.csv')
X_test     = pd.read_csv(SPLITS_V3 / 'test_features.csv')
y_test     = pd.read_csv(SPLITS_V3 / 'test_target.csv')
y_val_orig = pd.read_csv(SPLITS_V3 / 'val_target_original.csv')

print(f'X_train : {X_train.shape}   (expected 47 cols)')
print(f'X_val   : {X_val.shape}')
print(f'X_test  : {X_test.shape}')

assert X_train.shape[1] == 47, f'Expected 47 cols, got {X_train.shape[1]}'

v3_features = ['lag_16', 'lag_21', 'lag_35', 'rolling_mean_30', 'rolling_mean_28_safe', 'rolling_std_28_safe']
removed     = ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_mean_7', 'rolling_mean_14', 'rolling_mean_28', 'rolling_std_7']

print(f'\nv3 features present:')
for col in v3_features:
    print(f'  {col:<25}: {"YES ✓" if col in X_train.columns else "MISSING ✗"}')
print(f'Removed features absent:')
for col in removed:
    print(f'  {col:<25}: {"absent ✓" if col not in X_train.columns else "STILL PRESENT ✗"}')

## 1b. Encode Categorical Features

In [ ]:
object_cols    = X_train.select_dtypes(include=['object']).columns.tolist()
label_encoders = {}

for col in object_cols:
    le    = LabelEncoder()
    parts = [X_train[col], X_val[col]]
    if col in X_test.columns:
        parts.append(X_test[col])
    combined = pd.concat(parts, ignore_index=True).astype(str)
    le.fit(combined)

    X_train[col] = le.transform(X_train[col].astype(str)).astype(np.int32)
    X_val[col]   = le.transform(X_val[col].astype(str)).astype(np.int32)
    if col in X_test.columns:
        X_test[col] = le.transform(X_test[col].astype(str)).astype(np.int32)

    label_encoders[col] = le

remaining = X_train.select_dtypes(include=['object']).columns.tolist()
assert not remaining, f'Still has object columns: {remaining}'
print(f'LabelEncoded: {object_cols}')
print(f'X_train {X_train.shape} | X_val {X_val.shape} | X_test {X_test.shape}')

## 2. Metrics Helper

In [ ]:
y_test_orig = pd.DataFrame({'sales': np.expm1(y_test['sales_log'])})

def evaluate_metrics(y_true_log, y_pred_log, y_true_orig, label=''):
    y_pred_orig = np.clip(np.expm1(y_pred_log), 0, None)
    y_true_orig = np.clip(y_true_orig, 0, None)

    rmsle    = np.sqrt(mean_squared_log_error(y_true_orig, y_pred_orig))
    rmse     = np.sqrt(mean_squared_error(y_true_orig, y_pred_orig))
    mae      = mean_absolute_error(y_true_orig, y_pred_orig)
    rmse_log = np.sqrt(mean_squared_error(y_true_log, y_pred_log))

    if label:
        print(f'\nXGBoost — {label}:')
        print(f'  RMSLE              : {rmsle:.6f}')
        print(f'  RMSE (sales units) : {rmse:.4f}')
        print(f'  MAE  (sales units) : {mae:.4f}')
        print(f'  RMSE (log scale)   : {rmse_log:.6f}')

    return {'RMSLE': rmsle, 'RMSE': rmse, 'MAE': mae, 'RMSE_log': rmse_log}

## 3. Load Optuna Params & Train với GPU

In [ ]:
params_file = MODELS_DIR / 'xgboost_optuna_best_params.json'
assert params_file.exists(), f'Optuna params không tìm thấy: {params_file}'

with open(params_file, 'r') as f:
    best_optuna_params = json.load(f)

print('Optuna best params:')
for k, v in best_optuna_params.items():
    print(f'  {k:<20}: {v}')

In [ ]:
X_tv = pd.concat([X_train, X_val], ignore_index=True)
y_tv = pd.concat([y_train['sales_log'], y_val['sales_log']], ignore_index=True)

print(f'Train+Val: {X_tv.shape}')

best_params = {
    **best_optuna_params,
    'n_estimators': 1000,
    'objective':    'reg:squarederror',
    'tree_method':  'hist',
    'device':       'cuda',
    'random_state': 42,
    'n_jobs':       -1,
}

model_file = MODELS_DIR / 'xgboost_safe_lag_v2_model.pkl'

if model_file.exists():
    print(f'\nLoading saved model: {model_file.name}')
    best_model = joblib.load(model_file)
else:
    print('\nTraining XGBoost (GPU + safe lag v3 features)...')
    print('Params:', {k: v for k, v in best_params.items() if k not in ['n_jobs', 'random_state']})
    best_model = xgb.XGBRegressor(**best_params)
    best_model.fit(X_tv, y_tv, verbose=100)
    model_file.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_model, model_file)
    print(f'\nSaved: {model_file}')

## 4. Evaluation & So sánh

In [ ]:
val_metrics  = evaluate_metrics(
    y_val['sales_log'],
    best_model.predict(X_val),
    y_val_orig['sales'],
    'Val (Jul 2017) — safe lag v3'
)

test_metrics = evaluate_metrics(
    y_test['sales_log'],
    best_model.predict(X_test),
    y_test_orig['sales'],
    'Test (Aug 01–15) — safe lag v3'
)

print('\n' + '='*70)
print('SO SÁNH: XGBoost v1 (45 feat) vs v3 (47 feat, no distribution shift)')
print('='*70)
print(f'  Val  RMSLE v1 (safe_lag)   : 0.?????? (chạy XGBoost_training_safe_lag.ipynb để biết)')
print(f'  Val  RMSLE v3 (safe_lag_v3): {val_metrics["RMSLE"]:.6f}')
print(f'  Test RMSLE v1 (safe_lag)   : 0.?????? ')
print(f'  Test RMSLE v3 (safe_lag_v3): {test_metrics["RMSLE"]:.6f}')
print(f'  XGB toxic lags (baseline)  : Val=0.331299 | Test=0.370917')
print('='*70)

## 5. Save Model

In [ ]:
save_dir = NOTEBOOKS_DIR / '08_forecasting'
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, save_dir / 'xgb_safe_lag_v2_model.pkl')
print(f'Saved copy: {save_dir / "xgb_safe_lag_v2_model.pkl"}')
print(f'Main model: {MODELS_DIR / "xgboost_safe_lag_v2_model.pkl"}')
print(f'\nn_features_in_: {best_model.n_features_in_}  (expected 47)')
print(f'Feature names   : {list(X_train.columns)}')

# Lưu label encoders để forecasting notebook dùng
joblib.dump(label_encoders, save_dir / 'xgb_v2_label_encoders.pkl')
print(f'Label encoders  : {save_dir / "xgb_v2_label_encoders.pkl"}')